# Homework: classifying paragraphs by theme

In class we classified the **variants** of Frankenstein into *error*, *substantive* and *vocabulary*. Here we use **the same two methods on a different dataset**:

- the paragraphs in `data/ALL_paragraphs`: 150 paragraphs from three American novels of the 1920s
  (`CATH` = Willa Cather, *A Lost Lady*; `FITZ` = F. Scott Fitzgerald, *The Great Gatsby*; `HEMI` = Ernest Hemingway, *The Sun Also Rises*);
- three new categories: **social class**, **nature**, **love**.

The notebook has two parts:
1. **Zero-shot classification** with an NLI model, as in [zero_shot_variants_all.ipynb](../zero_shot_variants_all.ipynb);
2. **Few-shot prompting with an LLM**, as in [LLM-classification.ipynb](../LLM-classification.ipynb).

The code is (almost) the same as in those notebooks. What changes is **the data**, **the categories** and **the way we describe them** (hypotheses and definitions).

## The data and the ground truth

Each paragraph is a `.txt` file. We read all of them into one table, with one row per paragraph: `file` (the file name) and `text` (the paragraph).

The **ground truth** is a CSV file with two columns: `file` and `category`. Write its name in `GT_FILE`.
- `gt.csv` = **to be filled by you**: write *social class*, *nature* or *love* next to the paragraphs you have read. Leave the cell empty if no category fits.
- `gt_fake.csv` = produced by an LLM, for testing purposes.

Paragraphs without a category are classified anyway, but they are not used for the evaluation.

In [1]:
from pathlib import Path  # Provides tools for locating files and folders.
import pandas as pd  # Provides DataFrames for tables.
pd.set_option("display.max_colwidth", 120)  # Shows longer texts in table cells.

DATA_DIR = Path("../../data/ALL_paragraphs")  # The folder with the paragraphs.
GT_FILE = "gt_fake.csv"  # The file with our classification: gt.csv (yours) or gt_fake.csv (fake data).
CATEGORIES = ["social class", "nature", "love"]  # The categories of the homework.
GOLD_COL = "category"  # The column with our category.

# One row per file: its name and its text.
df = pd.DataFrame([{"file": path.name, "text": path.read_text(encoding="utf-8").strip()}
                   for path in sorted(DATA_DIR.glob("*.txt"))])

ours = pd.read_csv(GT_FILE)[["file", "category"]]
ours["category"] = ours["category"].str.strip().str.lower()  # "Nature " -> "nature"
ours = ours[ours["category"].isin(CATEGORIES)]  # Drops empty or unknown categories.
df = df.merge(ours, on="file", how="left")  # Paragraphs without a category get NaN (empty).

print(f"Loaded {len(df)} paragraphs from {DATA_DIR}, {df[GOLD_COL].notna().sum()} of them with a category in {GT_FILE}")
df[GOLD_COL].value_counts()

Loaded 150 paragraphs from ../../data/ALL_paragraphs, 112 of them with a category in gt_fake.csv


category
love            41
social class    37
nature          34
Name: count, dtype: int64

# Part 1. Zero-shot classification

**Tool:** [`facebook/bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli), the same NLI model as in class.

As a reminder: the model scores how strongly a **premise** *entails* a **hypothesis**. We write one hypothesis per category; the best-scoring hypothesis gives the predicted category.

| | Variants (in class) | Paragraphs (homework) |
|---|---|---|
| **Premise** | *In the manuscript, "kign" was replaced with "king".* | the paragraph itself |
| **Hypotheses** | *"kign" is a mistake…* / *"king" means something similar to "kign".* / … | *This text is about social class…* / *This text is about nature…* / *This text is about love…* |

## Build the premise
The premise is simply the text of the paragraph. We keep the function, as in class, so that you can change it (for example, to give the model only the first sentence).

In [2]:
def make_premise(row):
    # return row["text"].split(". ")[0]  # Only the first sentence.
    return row["text"]

df["premise"] = df.apply(make_premise, axis=1)
df["premise"].iloc[0]

'Thirty or forty years ago, in one of those grey towns along the Burlington railroad, which are so much greyer to-day than they were then, there was a house well known from Omaha to Denver for its hospitality and for a certain charm of atmosphere. Well known, that is to say, to the railroad aristocracy of that time; men who had to do with the railroad itself, or with one of the "land companies" which were its by-products. In those days it was enough to say of a man that he was "connected with the Burlington." There were the directors, the general managers, vice-presidents, superintendents, whose names we all knew; and their younger brothers or nephews were auditors, freight agents, departmental assistants. Everyone "connected" with the Road, even the large cattle and grain-shippers, had annual passes; they and their families rode about over the line a great deal. There were then two distinct social strata in the prairie States; the homesteaders and hand-workers who were there to make a

## Write the hypotheses
For the variants, the hypotheses changed with every variant, because they contained the two readings.
Here they are **the same for every paragraph**: each one describes a theme. We still write a function, as in class.

This is the part to experiment with: the words you choose to describe each category change the results.

In [3]:
def make_hypotheses(row):
    # Returns one hypothesis per category. Here they do not depend on the paragraph.
    return {
        "social class": "This text is about social class: money, wealth, status, rich and poor people.",
        "nature":       "This text is about nature: landscape, weather, plants or animals.",
        "love":         "This text is about love: romantic feelings, desire, couples or marriage.",
    }

for category, hypothesis in make_hypotheses(df.iloc[0]).items():
    print(f"{category:<13} {hypothesis}")

social class  This text is about social class: money, wealth, status, rich and poor people.
nature        This text is about nature: landscape, weather, plants or animals.
love          This text is about love: romantic feelings, desire, couples or marriage.


## Load the model and test one example
The first run downloads the model (~1.6 GB). It runs on CPU; a GPU is faster but not required.

In [4]:
from transformers import pipeline


classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

def classify(row):
    # Returns the score of each category for one paragraph.
    hypotheses = make_hypotheses(row)
    result = classifier(
        row["premise"],
        list(hypotheses.values()),  # The hypotheses are the candidate "labels".
        hypothesis_template="{}",   # They are already full sentences: nothing to add around them.
        multi_label=False,           # The categories compete: the scores sum to 1.
    )
    score_of = dict(zip(result["labels"], result["scores"]))  # hypothesis -> score
    return {category: round(score_of[h], 3) for category, h in hypotheses.items()}

# Test on two paragraphs.
for i in [5, 79]:
    row = df.iloc[i]
    print("Premise:", row["premise"][:200], "...")
    for category, score in classify(row).items():
        print(f"  {category:<13} {score:.3f}")

/home/elena/Github/DHBern/text-analysis-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4512.58it/s]


Premise: The wild roses were wide open and brilliant, the blue-eyed grass was in purple flower, and the silvery milkweed was just coming on. Birds and butterflies darted everywhere. All at once the breeze died ...
  social class  0.043
  nature        0.797
  love          0.160
Premise: His heart beat faster as Daisy’s white face came up to his own. He knew that when he kissed this girl, and forever wed his unutterable visions to her perishable breath, his mind would never romp again ...
  social class  0.044
  nature        0.160
  love          0.796


## Classify the whole corpus
For each paragraph we keep the **predicted category** and the **confidence** (score of the winner). This takes a few minutes: the paragraphs are much longer than the variants.

In [5]:
scores = pd.DataFrame([classify(row) for _, row in df.iterrows()], index=df.index)  # One row per paragraph, one column per category.

df["predicted"]  = scores.idxmax(axis=1)  # The category with the highest score.
df["confidence"] = scores.max(axis=1)     # Its score.
for category in scores.columns:
    df["score_" + category] = scores[category]  # Keeps all the scores, to look at them later.

df[["file", "text", "score_social class", "score_nature", "score_love", "predicted"]]

,file,text,score_social class,score_nature,score_love,predicted
0,CATH-ALL-23_01.txt,"Thirty or forty years ago, in one of those grey towns along the Burlington railroad, which are so much greyer to-day...",0.748,0.111,0.141,social class
1,CATH-ALL-23_02.txt,"The Forrester place, as every one called it, was not at all remarkable; the people who lived there made it seem much...",0.116,0.655,0.228,nature
2,CATH-ALL-23_03.txt,"To approach Captain Forrester's property, you had first to get over a wide, sandy creek which flowed along the easte...",0.161,0.618,0.221,nature
3,CATH-ALL-23_04.txt,"Mrs. Forrester was twenty-five years younger than her husband, and she was his second wife. He married her in Califo...",0.101,0.239,0.660,love
4,CATH-ALL-23_05.txt,When lunch time came they had done none of the things they meant to do. They had behaved like wild creatures all mor...,0.049,0.794,0.156,nature
...,...,...,...,...,...,...
145,HEMI-TSAR-26_46.txt,"The waiter seemed a little offended about the flowers of the Pyrenees, so I overtipped him. That made him happy. It ...",0.098,0.441,0.461,love
146,HEMI-TSAR-26_47.txt,"Next morning I tipped every one a little too much at the hotel to make more friends, and left on the morning train f...",0.139,0.199,0.662,love
147,HEMI-TSAR-26_48.txt,"I undressed in one of the bath-cabins, crossed the narrow line of beach and went into the water. I swam out, trying ...",0.153,0.515,0.333,nature
148,HEMI-TSAR-26_49.txt,That seemed to handle it. That was it. Send a girl off with one man. Introduce her to another to go off with him. No...,0.063,0.110,0.827,love


In [6]:
# How many paragraphs per category?
df["predicted"].value_counts()

predicted
love            75
nature          49
social class    26
Name: count, dtype: int64

In [7]:
# Uncertain cases deserve a human look (threshold is arbitrary: adjust it)
df[df["confidence"] < 0.6][["file", "text", "predicted", "confidence"]]

,file,text,predicted,confidence
6,CATH-ALL-23_07.txt,To this the boys made no reply. Such an assertion was absurd even to fish-mouthed Thad; his father's business depend...,social class,0.516
8,CATH-ALL-23_09.txt,Dr. Dennison set Niel's arm and took him home in his buggy. Home was not a pleasant place to go to; a frail egg-shel...,social class,0.516
9,CATH-ALL-23_10.txt,All this while the town of Sweet Water was changing. Its future no longer looked bright. Successive crop failures ha...,nature,0.410
10,CATH-ALL-23_11.txt,"Niel Herbert's father was one of the first failures to be crowded to the wall. He closed his little house, sent his ...",love,0.520
12,CATH-ALL-23_13.txt,"A little later Niel went plunging down the hill, looking exultantly into the streak of red sunset. Oh, the winter wo...",love,0.493
...,...,...,...,...
142,HEMI-TSAR-26_43.txt,"“That was rather good. Ashley, chap she got the title from, was a sailor, you know. Ninth baronet. When he came home...",love,0.528
144,HEMI-TSAR-26_45.txt,I sat in one of the wicker chairs and leaned back comfortably. The waiter was in no hurry to come. The white-paper a...,love,0.456
145,HEMI-TSAR-26_46.txt,"The waiter seemed a little offended about the flowers of the Pyrenees, so I overtipped him. That made him happy. It ...",love,0.461
147,HEMI-TSAR-26_48.txt,"I undressed in one of the bath-cabins, crossed the narrow line of beach and went into the water. I swam out, trying ...",nature,0.515


## Evaluate
Compare the model with **your** judgement, on the paragraphs that have a category in the ground truth.
- **Accuracy**: share of paragraphs where model and human agree.
- **Confusion matrix**: rows = human label, columns = model label. Off-diagonal cells show *which* categories the model confuses.

In [8]:
labelled = df[df[GOLD_COL].notna()]  # Only the paragraphs with a category in the ground truth.
accuracy = (labelled["predicted"] == labelled[GOLD_COL]).mean()
print(f"Accuracy: {accuracy:.0%} on {len(labelled)} paragraphs")
pd.crosstab(labelled[GOLD_COL], labelled["predicted"], rownames=["human"], colnames=["model"])

Accuracy: 84% on 112 paragraphs


model,love,nature,social class
human,,,
love,37,2,2
nature,0,34,0
social class,12,2,23


## Save the results

In [9]:
df.drop(columns="premise").to_csv("paragraphs_classified_zero_shot.csv", index=False)
print("Saved to paragraphs_classified_zero_shot.csv")

Saved to paragraphs_classified_zero_shot.csv


# Part 2. Few-shot prompting with an LLM

We send our questions to GPUStack.

### Connect to the university's model
The key, the address and the name of the model are read from the `.env` file, as in class (the one in `session07` is found automatically).

In [10]:
import requests  # Sends requests over the internet.
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
MODEL_NAME = os.getenv("MODEL_NAME")
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",  # The key, sent with every request.
    "Content-Type": "application/json",    # We send our request in JSON format.
}

## Examples and test set

We split the paragraphs in two groups:
- **examples**: a few paragraphs per category, taken from the ground truth and shown to the model in the instructions;
- **test set**: all the others. The model classifies them, and we check its answers against ours (where we have one).

A paragraph used as an example must not be in the test set, otherwise we would be giving the model the answer.

In [11]:
N_EXAMPLES = 2  # Examples per category. 0 = zero-shot.

paragraphs = df[["file", "text", GOLD_COL]]  # Only the data, without the results of Part 1.
ground_truth = paragraphs[paragraphs[GOLD_COL].notna()]
examples = ground_truth.groupby("category").sample(N_EXAMPLES, random_state=1) if N_EXAMPLES else ground_truth.iloc[:0]
test = paragraphs.drop(examples.index)

print(len(examples), "examples,", len(test), "paragraphs in the test set")
examples[["file", "text", "category"]]

6 examples, 144 paragraphs in the test set


,file,text,category
17,CATH-ALL-23_18.txt,"Ellinger glanced at her averted head, and his heavy lips twitched in a smile at one corner. The quality of her voice...",love
15,CATH-ALL-23_16.txt,"Her eyes fell. ""Good-night,"" she said faintly. As she turned quickly away, the train of her velvet dress caught the ...",love
114,HEMI-TSAR-26_15.txt,We ate the sandwiches and drank the Chablis and watched the country out of the window. The grain was just beginning ...,nature
115,HEMI-TSAR-26_16.txt,"After a while we came out of the mountains, and there were trees along both sides of the road, and a stream and ripe...",nature
100,HEMI-TSAR-26_01.txt,"Robert Cohn was a member, through his father, of one of the richest Jewish families in New York, and through his mot...",social class
75,FITZ-TGG-25_26.txt,"Cody was fifty years old then, a product of the Nevada silver fields, of the Yukon, of every rush for metal since se...",social class


## The instructions (the *prompt*)

The model only knows what we write here. Compare the definitions with the hypotheses of Part 1: they describe the same categories.

This is the part to experiment with: change the definitions and see how the results change.

In [12]:
DEFINITIONS = """\
- social class: the paragraph is mainly about money, wealth, social status, the difference between rich and poor people.
- nature: the paragraph is mainly about landscape, weather, plants or animals.
- love: the paragraph is mainly about romantic feelings, desire, couples or marriage."""

def show_paragraph(row):
    # Describes one paragraph as text.
    return f'Paragraph: "{row["text"]}"'

instructions = f"""You are an expert in literary studies. You classify paragraphs from American novels of the 1920s.
Classify each paragraph in one of these categories, according to its main theme:
{DEFINITIONS}

Answer in this format: category | one short sentence explaining why."""

if len(examples):
    instructions += "\n\nExamples:\n" + "\n".join(
        f'{show_paragraph(r)}\nCategory: {r["category"]}' for _, r in examples.iterrows())

print(instructions)

You are an expert in literary studies. You classify paragraphs from American novels of the 1920s.
Classify each paragraph in one of these categories, according to its main theme:
- social class: the paragraph is mainly about money, wealth, social status, the difference between rich and poor people.
- nature: the paragraph is mainly about landscape, weather, plants or animals.
- love: the paragraph is mainly about romantic feelings, desire, couples or marriage.

Answer in this format: category | one short sentence explaining why.

Examples:
Paragraph: "Ellinger glanced at her averted head, and his heavy lips twitched in a smile at one corner. The quality of her voice had changed, and he knew the change. They went spinning along the curves of the winding road, saying not a word. Mrs. Forrester sat with her head bent forward, her face half hidden in her muff. At last she told him to stop. To the right of the road he saw a thicket. Behind it a dry watercourse wound into the bluffs. The top

## Ask the model about one paragraph

The functions are the same as in class: `ask` sends one paragraph to the model, `read_category` extracts the category from the answer.

In [15]:
import re  # Regular expressions, to remove the "thinking" part.
import time  # To wait between requests.

def ask(row):
    # Sends one paragraph to the model and returns its answer (a text).
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "system", "content": instructions},
                     {"role": "user", "content": show_paragraph(row)}],
        "temperature": 0,
        "max_tokens": 1000,
    }
    for attempt in range(5):
        try:
            response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120)
        except requests.exceptions.RequestException as error:  # No connection, or the server is too slow.
            print("Connection problem, waiting 20 seconds:", str(error)[:100])
            time.sleep(20)
            continue
        if response.status_code == 429 or response.status_code >= 500:  # Too many requests, or server busy.
            print("Server busy, waiting 20 seconds:", response.status_code)
            time.sleep(20)
            continue
        response.raise_for_status()  # Other problems (e.g. wrong key or URL): stop and show the error.
        answer = response.json()["choices"][0]["message"]["content"] or ""
        return re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    return ""

def read_category(answer):
    # Extracts the category from an answer such as "nature | a description of a landscape".
    first_part = answer.split("|")[0].lower()
    for category in CATEGORIES:
        if category in first_part:
            return category
    return "unknown"

row = test.iloc[2]
print(show_paragraph(row))
answer = ask(row)
print("\nModel:", answer)
print("Category:", read_category(answer), "| Ours:", row["category"])

Paragraph: "To approach Captain Forrester's property, you had first to get over a wide, sandy creek which flowed along the eastern edge of the town. Crossing this by the foot-bridge or the ford, you entered the Captain's private lane, bordered by Lombardy poplars, with wide meadows lying on either side. Just at the foot of the hill on which the house sat, one crossed a second creek by the stout wooden road-bridge. This stream traced artless loops and curves through the broad meadows that were half pasture land, half marsh. Any one but Captain Forrester would have drained the bottom land and made it into highly productive fields. But he had selected this place long ago because it looked beautiful to him, and he happened to like the way the creek wound through his pasture, with mint and joint-grass and twinkling willows along its banks. He was well off for those times, and he had no children. He could afford to humour his fancies."

Model: nature | The paragraph mainly describes the cree

## Classify the test set

One request per paragraph: this takes a few seconds per paragraph.

In [14]:
answers = []
for i, (_, row) in enumerate(test.iterrows(), start=1):
    answers.append(ask(row))
    print(i, "/", len(test), end="\r")  # Progress.

test = test.assign(answer=answers)
test["predicted"] = test["answer"].apply(read_category)
test["reason"] = test["answer"].str.split("|").str[-1].str.strip()

test[["file", "text", "category", "predicted", "reason"]]

KeyboardInterrupt: 

## Evaluation

- **Accuracy**: share of paragraphs where the model agrees with us.
- **Confusion matrix**: rows = our category, columns = the model's category.

Only the paragraphs with a category in the ground truth are evaluated.

In [ ]:
labelled = test[test["category"].notna()]  # Only the paragraphs with a category in the ground truth.
accuracy = (labelled["predicted"] == labelled["category"]).mean()
print(f"Accuracy: {accuracy:.0%} on {len(labelled)} paragraphs")
pd.crosstab(labelled["category"], labelled["predicted"], rownames=["ours"], colnames=["model"])

In [ ]:
# The disagreements, with the model's explanation: who is right?
labelled.loc[labelled["predicted"] != labelled["category"], ["file", "text", "category", "predicted", "reason"]]

In [ ]:
# Saves the classified paragraphs, with the model's answers, to a CSV file.
# The model name is part of the file name, so results of different models are kept separate.
OUTPUT_FILE = "paragraphs_classified_llm_" + re.sub(r"[^A-Za-z0-9.-]", "_", MODEL_NAME) + ".csv"  # "/" or ":" are not allowed in file names.
test.to_csv(OUTPUT_FILE, index=False)
print("Saved to", OUTPUT_FILE)

## Exercises

1. **Your ground truth.** Fill `gt.csv` with your own categories (at least 10 paragraphs per category), set `GT_FILE = "gt.csv"` and run the whole notebook again. How do the results change compared to `gt_fake.csv`?
2. **Compare the two methods.** Which one agrees more with you? Open the two CSV files: on which paragraphs do the NLI model and the LLM disagree?
3. **Hypotheses and definitions.** Rewrite the hypotheses (Part 1) and the definitions (Part 2), for example *"This text describes a landscape."* instead of *"This text is about nature…"*. Which wording works best?
4. **None of the three.** Many paragraphs are about none of our themes, and some are about two (e.g. love *and* social class in *The Great Gatsby*). The models must still choose one. Add a fourth category, *other*, in `CATEGORIES`, in the hypotheses and in the definitions. What happens?
5. **Zero-shot vs few-shot.** In Part 2, set `N_EXAMPLES = 0`, then `N_EXAMPLES = 4`. Does accuracy change?